# 📋 規格書（請直接在這一格修改）

學號：411312537姓名：林彣媞卷別：B

## 1. 系統目的（一句話）
提供學生透過自然語言進行校園咖啡廳之飲品預約、取消、庫存與營收統計查詢的智慧化服務系統。
## 2. 使用者會怎麼說（至少 5 句）
1. 我想訂兩杯拿鐵
2. 查詢抹茶歐蕾庫存
3. 取消剛剛的拿鐵
4. 今天各品項賣得怎樣
5. 我要一杯美式咖啡

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| menu | item_name | TEXT (Primary Key) | 品項名稱 |
| menu | price | INTEGER | 單價（元） |
| menu | stock | INTEGER | 今日庫存（杯） |
| orders | order_id | INTEGER (Primary Key AutoIncrement) | 訂單編號 |
| orders | student_name | TEXT | 學生姓名 |
| orders | item_name | TEXT | 品項名稱 |
| orders | quantity | INTEGER | 訂購數量 |
| orders | status | TEXT | 訂單狀態（正常 / 已取消） |

初始資料：
- 美式咖啡：單價 50 元，庫存 5 杯
- 拿鐵：單價 65 元，庫存 4 杯
- 抹茶歐蕾：單價 70 元，庫存 3 杯
## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,品項,數量`
- 動作只能是：訂購、取消、查詢、統計、未知
- 品項只能是：美式咖啡、拿鐵、抹茶歐蕾、無
- 數量的規則：若使用者未提及數量則預設為 1，介於 1 至 3 之間
- 無法判斷時輸出：`未知,無,1`
- AI 不可以做的事：直接修改資料庫、略過格式自行發揮、捏造不存在的品項

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 訂購 | 庫存足夠、個人今日總杯數 $\le 3$、數量 $1 \sim 3$ | 成功訂購 [品項] [數量] 杯，總金額 [金額] 元 | 扣除對應庫存，新增一筆訂單記錄 |
| 取消 | 該學生有該品項之有效訂單 | 已成功取消您最近一筆 [品項] 訂單 | 釋放庫存，該訂單狀態改為「已取消」 |
| 查詢庫存 | 品項存在於菜單中 | [品項] 目前剩餘庫存為 [數量] 杯 | 無 |
| 統計查詢 | 使用者詢問銷售或營收狀況 | 列出所有品項的單價、初始庫存、已售杯數、剩餘杯數、總營收 |

## 6. 例外情況（至少 3 個）
1. 庫存不足：當訂購數量大於剩餘庫存時，系統拒絕並回報庫存不足。
2. 超過個人上限：當學生今天已訂購總杯數達 3 杯再嘗試加購時，系統拒絕。
3. 查無可取消訂單：當學生要求取消但查無對應的有效訂單時，系統提示找不到訂單。

## 7. 測試清單
**7-1 AI 解讀測試（8～10 句）**

| # | 類型 | 使用者說的話 | 預期動作 | 預期品項 | 預期數量 |
|---|---|---|---|---|---|
| 1 | 正常 | 我想要兩杯拿鐵 | 訂購 | 拿鐵 | 2 |
| 2 | 正常 | 查詢美式咖啡庫存 | 查詢 | 美式咖啡 | 1 |
| 3 | 正常 | 取消拿鐵 | 取消 | 拿鐵 | 1 |
| 4 | 正常 | 今天各品項賣得怎樣 | 統計 | 無 | 1 |
| 5 | 正常 | 幫我訂一杯抹茶歐蕾 | 訂購 | 抹茶歐蕾 | 1 |
| 6 | 錯誤 | 訂購 5 杯美式咖啡 | 訂購 | 美式咖啡 | 5 |
| 7 | 錯誤 | 我要喝珍奶 | 訂購 | 未知 | 1 |
| 8 | 邊界 | 拿鐵 | 查詢 | 拿鐵 | 1 |

**7-2 流程測試（8～10 步，依序執行）**

| # | 姓名 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 王小明 | 我要訂 2 杯拿鐵 | 成功訂購，金額 130 元 |
| 2 | 王小明 | 我要訂 2 杯拿鐵 | 拒絕（超過個人 3 杯上限） |
| 3 | 李大華 | 我要訂 4 杯美式咖啡 | 拒絕（庫存不足或數量超過限制） |
| 4 | 李大華 | 我要訂 2 杯美式咖啡 | 成功訂購，金額 100 元 |
| 5 | 李大華 | 查詢美式咖啡庫存 | 剩餘 3 杯 |
| 6 | 李大華 | 取消美式咖啡 | 成功取消，庫存恢復 |
| 7 | 王小明 | 查詢拿鐵庫存 | 剩餘 2 杯 |
| 8 | 店長 | 今天各品項賣得怎樣 | 顯示完整的統計報表與營收 |

（若有改用其他模型或有測試未通過，請在此說明原因）

In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
def init_db():
    conn = sqlite3.connect('coffee_shop.db')
    cursor = conn.cursor()
    try:
        cursor.execute("DROP TABLE IF EXISTS orders")
        cursor.execute("DROP TABLE IF EXISTS menu")

        # 建立菜單表
        cursor.execute("""
            CREATE TABLE menu (
                item_name TEXT PRIMARY KEY,
                price INTEGER NOT NULL,
                stock INTEGER NOT NULL
            )
        """)

        # 建立訂單表
        cursor.execute("""
            CREATE TABLE orders (
                order_id INTEGER PRIMARY KEY AUTOINCREMENT,
                customer_name TEXT NOT NULL,
                item_name TEXT NOT NULL,
                quantity INTEGER NOT NULL,
                status TEXT NOT NULL
            )
        """)

        # 寫入初始菜單資料
        initial_menu = [
            ("美式咖啡", 50, 5),
            ("拿鐵", 65, 4),
            ("抹茶歐蕾", 70, 3)
        ]
        cursor.executemany("INSERT INTO menu VALUES (?, ?, ?)", initial_menu)
        conn.commit()
        print("✅ 資料庫初始化完成！")
    finally:
        conn.close()

init_db()

✅ 資料庫初始化完成！


In [3]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
def show_tables():
    conn = sqlite3.connect('coffee_shop.db')
    try:
        print("--- 菜單表 (menu) ---")
        df_menu = pd.read_sql_query("SELECT * FROM menu", conn)
        display(df_menu)

        print("\n--- 訂單表 (orders) ---")
        df_orders = pd.read_sql_query("SELECT * FROM orders", conn)
        display(df_orders)
    finally:
        conn.close()

show_tables()

--- 菜單表 (menu) ---


,item_name,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3



--- 訂單表 (orders) ---


,order_id,customer_name,item_name,quantity,status


In [4]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 品項, 數量, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
ai_cache = {}

def parse_user_intent(user_text):
    if user_text in ai_cache:
        return ai_cache[user_text]

    prompt = f"""
你是一個校園咖啡廳系統的意圖解析員。
請從以下使用者的話中抽出「動作」、「品項」、「數量」。

允許的動作：訂購, 取消, 查詢, 統計, 未知
允許的品項：美式咖啡, 拿鐵, 抹茶歐蕾, 無, 未知

規則：
1. 若動作為「統計」（如查詢賣得怎樣、今日銷售），品項請填「無」。
2. 數量請轉換為阿拉伯數字整數，若未明確說明數量，預設為 1。
3. 請嚴格只輸出 `動作,品項,數量` 的格式，不要有額外說明。

使用者說的話："{user_text}"
"""

    raw_response = ""
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            raw_response = response.choices[0].message.content.strip()
            time.sleep(2)
            break
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                print("⚠️ 達到 Rate Limit，等待 30 秒...")
                time.sleep(30)
            else:
                raw_response = "未知,未知,1"
                break

    # 程式驗證與安全後處理（不直接信任 AI）
    valid_actions = ["訂購", "取消", "查詢", "統計"]
    valid_items = ["美式咖啡", "拿鐵", "抹茶歐蕾"]

    action = "未知"
    for act in valid_actions:
        if act in raw_response:
            action = act
            break

    item = "未知"
    if action == "統計":
        item = "無"
    else:
        for it in valid_items:
            if it in raw_response:
                item = it
                break

    # 正規表示式提取數量
    qty_match = re.search(r'\d+', raw_response)
    quantity = int(qty_match.group()) if qty_match else 1

    result = (action, item, quantity, raw_response)
    ai_cache[user_text] = result
    return result

In [5]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================
test_cases = [
    ("我要兩杯拿鐵", "訂購", "拿鐵", 2),
    ("幫我取消抹茶歐蕾", "取消", "抹茶歐蕾", 1),
    ("請問美式咖啡還有嗎", "查詢", "美式咖啡", 1),
    ("今天各品項賣得怎樣", "統計", "無", 1),
    ("我要訂 3 杯美式咖啡", "訂購", "美式咖啡", 3),
    ("拿鐵查詢剩餘", "查詢", "拿鐵", 1),
    ("幫我訂珍奶", "未知", "未知", "*"),
    ("今天天氣真好", "未知", "未知", "*")
]

passed_count = 0
print("=== 開始 AI 解讀測試 ===\n")

for i, (text, exp_act, exp_item, exp_qty) in enumerate(test_cases, 1):
    act, item, qty, raw = parse_user_intent(text)

    act_ok = (act == exp_act)
    item_ok = (item == exp_item)
    qty_ok = (exp_qty == "*") or (qty == exp_qty)

    is_pass = act_ok and item_ok and qty_ok
    if is_pass:
        passed_count += 1

    print(f"測試 {i}: 「{text}」")
    print(f"  └ 原始回覆: {raw}")
    print(f"  └ 系統解讀: 動作={act}, 品項={item}, 數量={qty}")
    print(f"  └ 結果: {'✅ 通過' if is_pass else '❌ 失敗'}\n")

print(f"🎯 測試完成，共通過 {passed_count}/{len(test_cases)} 句")

=== 開始 AI 解讀測試 ===

測試 1: 「我要兩杯拿鐵」
  └ 原始回覆: 訂購,拿鐵,2
  └ 系統解讀: 動作=訂購, 品項=拿鐵, 數量=2
  └ 結果: ✅ 通過

測試 2: 「幫我取消抹茶歐蕾」
  └ 原始回覆: 取消,抹茶歐蕾,1
  └ 系統解讀: 動作=取消, 品項=抹茶歐蕾, 數量=1
  └ 結果: ✅ 通過

測試 3: 「請問美式咖啡還有嗎」
  └ 原始回覆: 查詢,美式咖啡,1
  └ 系統解讀: 動作=查詢, 品項=美式咖啡, 數量=1
  └ 結果: ✅ 通過

測試 4: 「今天各品項賣得怎樣」
  └ 原始回覆: 統計,無,1
  └ 系統解讀: 動作=統計, 品項=無, 數量=1
  └ 結果: ✅ 通過

測試 5: 「我要訂 3 杯美式咖啡」
  └ 原始回覆: 訂購,美式咖啡,3
  └ 系統解讀: 動作=訂購, 品項=美式咖啡, 數量=3
  └ 結果: ✅ 通過

測試 6: 「拿鐵查詢剩餘」
  └ 原始回覆: 查詢,拿鐵,1
  └ 系統解讀: 動作=查詢, 品項=拿鐵, 數量=1
  └ 結果: ✅ 通過

測試 7: 「幫我訂珍奶」
  └ 原始回覆: 訂購,未知,1
  └ 系統解讀: 動作=訂購, 品項=未知, 數量=1
  └ 結果: ❌ 失敗

測試 8: 「今天天氣真好」
  └ 原始回覆: 未知,無,1
  └ 系統解讀: 動作=未知, 品項=未知, 數量=1
  └ 結果: ✅ 通過

🎯 測試完成，共通過 7/8 句


In [6]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
def process_request(customer_name, user_text):
    action, item_name, quantity, _ = parse_user_intent(user_text)

    # 意圖或品項無法辨識
    if action == "未知" or (action in ["訂購", "取消", "查詢"] and item_name == "未知"):
        conn = sqlite3.connect('coffee_shop.db')
        try:
            df = pd.read_sql_query("SELECT item_name, price FROM menu", conn)
            items_str = "、".join([f"{r['item_name']}({r['price']}元)" for _, r in df.iterrows()])
            return f"❓ 無法理解您的意圖。目前提供品項有：{items_str}"
        finally:
            conn.close()

    conn = sqlite3.connect('coffee_shop.db')
    cursor = conn.cursor()
    try:
        # A3: 統計查詢 (LEFT JOIN + GROUP BY)
        if action == "統計":
            query = """
                SELECT
                    m.item_name AS 品項,
                    m.price AS 單價,
                    m.stock AS 剩餘庫存,
                    COALESCE(SUM(CASE WHEN o.status = '已訂購' THEN o.quantity ELSE 0 END), 0) AS 已售杯數,
                    COALESCE(SUM(CASE WHEN o.status = '已訂購' THEN o.quantity ELSE 0 END), 0) * m.price AS 營收
                FROM menu m
                LEFT JOIN orders o ON m.item_name = o.item_name
                GROUP BY m.item_name, m.price, m.stock
            """
            df_stat = pd.read_sql_query(query, conn)
            res_str = "📊 **今日銷售統計報告**：\n"
            for _, r in df_stat.iterrows():
                res_str += f"- {r['品項']}: 單價 {r['單價']}元 | 剩餘 {r['剩餘庫存']}杯 | 已售 {r['已售杯數']}杯 | 營收 {r['營收']}元\n"
            return res_str.strip()

        # 查詢庫存
        if action == "查詢":
            cursor.execute("SELECT stock FROM menu WHERE item_name = ?", (item_name,))
            row = cursor.fetchone()
            if row is not None:
                return f"🔍 目前【{item_name}】剩餘庫存為 {row[0]} 杯。"
            return f"❌ 找不到品項：{item_name}"

        # 訂購與取消需檢查姓名
        if not customer_name or not customer_name.strip():
            return "⚠️ 進行「訂購」或「取消」操作時，請務必輸入您的姓名！"

        # A1: 檢查單次數量限制 (1~3 杯)
        if quantity < 1 or quantity > 3:
            return f"❌ 訂購失敗：單次數量限制為 1 ~ 3 杯（您輸入了 {quantity} 杯）。"

        if action == "訂購":
            # A2: 檢查每名學生當日上限 3 杯
            cursor.execute("""
                SELECT COALESCE(SUM(quantity), 0)
                FROM orders
                WHERE customer_name = ? AND status = '已訂購'
            """, (customer_name,))
            user_bought = cursor.fetchone()[0]

            if user_bought + quantity > 3:
                return f"❌ 訂購失敗：每人每天最多訂購 3 杯。您已訂購 {user_bought} 杯，無法再訂購 {quantity} 杯。"

            # 檢查庫存與單價
            cursor.execute("SELECT price, stock FROM menu WHERE item_name = ?", (item_name,))
            row = cursor.fetchone()
            if row is None:
                return f"❌ 無此品項：{item_name}"

            price, stock = row
            if stock < quantity:
                return f"❌ 訂購失敗：【{item_name}】庫存不足（剩餘 {stock} 杯，需求 {quantity} 杯）。"

            # 扣減庫存並寫入訂單
            cursor.execute("UPDATE menu SET stock = stock - ? WHERE item_name = ?", (quantity, item_name))
            cursor.execute("INSERT INTO orders (customer_name, item_name, quantity, status) VALUES (?, ?, ?, '已訂購')",
                           (customer_name, item_name, quantity))
            conn.commit()
            total_price = price * quantity
            return f"🎉 訂購成功！{customer_name} 訂購【{item_name}】{quantity} 杯，共 {total_price} 元。"

        if action == "取消":
            cursor.execute("""
                SELECT order_id, quantity FROM orders
                WHERE customer_name = ? AND item_name = ? AND status = '已訂購'
                ORDER BY order_id DESC LIMIT 1
            """, (customer_name, item_name))
            row = cursor.fetchone()
            if row is None:
                return f"❌ 取消失敗：查無 {customer_name} 訂購【{item_name}】的有效紀錄。"

            order_id, qty = row
            cursor.execute("UPDATE orders SET status = '已取消' WHERE order_id = ?", (order_id,))
            cursor.execute("UPDATE menu SET stock = stock + ? WHERE item_name = ?", (qty, item_name))
            conn.commit()
            return f"🗑️ 取消成功！已為 {customer_name} 取消【{item_name}】{qty} 杯並釋出庫存。"

    finally:
        conn.close()

In [7]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
init_db()  # 測試前重置資料庫狀態

flow_tests = [
    ("小明", "我要 2 杯拿鐵"),
    ("小明", "我要 2 杯美式咖啡"),
    ("小華", "我要 5 杯美式咖啡"),
    ("小華", "我要 3 杯抹茶歐蕾"),
    ("小英", "訂 1 杯抹茶歐蕾"),
    ("小明", "取消拿鐵"),
    ("小英", "查詢美式咖啡庫存"),
    ("店長", "今天銷售狀況如何")
]

print("=== 開始流程測試 ===\n")
for i, (name, text) in enumerate(flow_tests, 1):
    reply = process_request(name, text)
    print(f"步驟 {i} [{name}]: 「{text}」")
    print(f"  └ 系統回應: {reply}\n")

print("=== 測試完成後的資料表狀態 ===")
show_tables()

✅ 資料庫初始化完成！
=== 開始流程測試 ===

步驟 1 [小明]: 「我要 2 杯拿鐵」
  └ 系統回應: 🎉 訂購成功！小明 訂購【拿鐵】2 杯，共 130 元。

步驟 2 [小明]: 「我要 2 杯美式咖啡」
  └ 系統回應: ❌ 訂購失敗：每人每天最多訂購 3 杯。您已訂購 2 杯，無法再訂購 2 杯。

步驟 3 [小華]: 「我要 5 杯美式咖啡」
  └ 系統回應: ❌ 訂購失敗：單次數量限制為 1 ~ 3 杯（您輸入了 5 杯）。

步驟 4 [小華]: 「我要 3 杯抹茶歐蕾」
  └ 系統回應: 🎉 訂購成功！小華 訂購【抹茶歐蕾】3 杯，共 210 元。

步驟 5 [小英]: 「訂 1 杯抹茶歐蕾」
  └ 系統回應: ❌ 訂購失敗：【抹茶歐蕾】庫存不足（剩餘 0 杯，需求 1 杯）。

步驟 6 [小明]: 「取消拿鐵」
  └ 系統回應: 🗑️ 取消成功！已為 小明 取消【拿鐵】2 杯並釋出庫存。

步驟 7 [小英]: 「查詢美式咖啡庫存」
  └ 系統回應: 🔍 目前【美式咖啡】剩餘庫存為 5 杯。

步驟 8 [店長]: 「今天銷售狀況如何」
  └ 系統回應: 📊 **今日銷售統計報告**：
- 抹茶歐蕾: 單價 70元 | 剩餘 0杯 | 已售 3杯 | 營收 210元
- 拿鐵: 單價 65元 | 剩餘 4杯 | 已售 0杯 | 營收 0元
- 美式咖啡: 單價 50元 | 剩餘 5杯 | 已售 0杯 | 營收 0元

=== 測試完成後的資料表狀態 ===
--- 菜單表 (menu) ---


,item_name,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,0



--- 訂單表 (orders) ---


,order_id,customer_name,item_name,quantity,status
0,1,小明,拿鐵,2,已取消
1,2,小華,抹茶歐蕾,3,已訂購


In [8]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
demo = gr.Interface(
    fn=process_request,
    inputs=[
        gr.Textbox(label="您的姓名（訂購/取消必填）", placeholder="例如：張小明"),
        gr.Textbox(label="請輸入您的需求", placeholder="例如：我要兩杯拿鐵 / 請問美式剩幾杯 / 今天賣得怎樣")
    ],
    outputs=gr.Textbox(label="系統回應"),
    title="☕ 校園咖啡廳預約取餐系統",
    description="支援預約取餐、取消訂單、剩餘庫存查詢與銷售統計報告。"
)

if IN_COLAB:
    demo.launch(share=True)
else:
    demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c546acdaf09585acbb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
